In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    classification_report
)
import xgboost as xgb

# 1. Đọc dữ liệu
print("Đang đọc dữ liệu train.csv...")
train = pd.read_csv("train.csv")

target_col = "is_fraud"
id_col = "id"
# Bỏ target_col, id_col và fold (nếu đã có)
ignore_cols = [target_col, id_col, "fold"]
feature_cols = [c for c in train.columns if c not in ignore_cols]

X = train[feature_cols].copy()
y = train[target_col].astype(int)

# 2. Xử lý các cột categorical: XGBoost hỗ trợ trực tiếp dtype 'category' với enable_categorical=True
cat_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].astype("category")

# 3. Tách tập Train (80%) và Validation (20%) có stratify theo nhãn y
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Kích thước X_train: {X_train.shape}, X_val: {X_val.shape}")
print(f"Tỉ lệ fraud train: {y_train.mean():.4f}, val: {y_val.mean():.4f}")

Đang đọc dữ liệu train.csv...
Kích thước X_train: (559968, 24), X_val: (139992, 24)
Tỉ lệ fraud train: 0.0167, val: 0.0167


In [2]:
# 4. Tính toán scale_pos_weight giải quyết mất cân bằng lớp (negative / positive)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Tỉ lệ scale_pos_weight: {scale_pos_weight:.2f}")

# 5. Khởi tạo mô hình XGBClassifier
xgb_model = xgb.XGBClassifier(
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",          # Tối ưu AUC-PR cho bài toán fraud imbalanced
    early_stopping_rounds=100,
    enable_categorical=True,      # Tự động xử lý biến category của pandas
    tree_method="hist",           # Huấn luyện cực nhanh bằng histogram
    random_state=42,
    n_jobs=-1
)

# 6. Huấn luyện mô hình với Early Stopping
print("Bắt đầu huấn luyện XGBoost...")
xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=100
)

# 7. Đánh giá 4 chỉ số trên tập Validation
val_prob = xgb_model.predict_proba(X_val)[:, 1]
threshold = 0.5
val_pred = (val_prob >= threshold).astype(int)

roc_auc = roc_auc_score(y_val, val_prob)
auc_pr = average_precision_score(y_val, val_prob)
precision = precision_score(y_val, val_pred, zero_division=0)
recall = recall_score(y_val, val_pred, zero_division=0)

print("\n" + "="*40)
print("--- ĐÁNH GIÁ TRÊN TẬP VALIDATION (0.2) ---")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"AUC-PR    : {auc_pr:.4f}")
print(f"Precision : {precision:.4f} (với threshold = {threshold})")
print(f"Recall    : {recall:.4f} (với threshold = {threshold})")
print("="*40)

print("\nBáo cáo chi tiết Classification Report:")
print(classification_report(y_val, val_pred, digits=4))

Tỉ lệ scale_pos_weight: 58.88
Bắt đầu huấn luyện XGBoost...
[0]	validation_0-aucpr:0.21060	validation_1-aucpr:0.21150
[100]	validation_0-aucpr:0.50969	validation_1-aucpr:0.48008
[200]	validation_0-aucpr:0.54821	validation_1-aucpr:0.49109
[300]	validation_0-aucpr:0.57884	validation_1-aucpr:0.49072
[320]	validation_0-aucpr:0.58439	validation_1-aucpr:0.48959

--- ĐÁNH GIÁ TRÊN TẬP VALIDATION (0.2) ---
ROC-AUC   : 0.9420
AUC-PR    : 0.4918
Precision : 0.1180 (với threshold = 0.5)
Recall    : 0.8323 (với threshold = 0.5)

Báo cáo chi tiết Classification Report:
              precision    recall  f1-score   support

           0     0.9968    0.8944    0.9428    137654
           1     0.1180    0.8323    0.2068      2338

    accuracy                         0.8933    139992
   macro avg     0.5574    0.8634    0.5748    139992
weighted avg     0.9821    0.8933    0.9305    139992



In [3]:
# 8. Dự đoán trên tập test.csv và xuất file submission
try:
    print("Đang đọc dữ liệu test.csv...")
    test = pd.read_csv("test.csv")
    X_test = test[feature_cols].copy()
    
    # Định dạng các cột phân loại giống hệt train
    for col in cat_cols:
        if col in X_test.columns:
            X_test[col] = X_test[col].astype("category")
    
    print("Đang dự đoán xác suất trên test...")
    test_prob = xgb_model.predict_proba(X_test)[:, 1]
    
    submission = pd.DataFrame({
        id_col: test[id_col],
        target_col: test_prob
    })
    submission.to_csv("submission_xgb.csv", index=False)
    print("Đã lưu kết quả dự đoán vào submission_xgb.csv thành công!")
except Exception as e:
    print("Lỗi khi dự đoán test:", e)

Đang đọc dữ liệu test.csv...
Đang dự đoán xác suất trên test...
Đã lưu kết quả dự đoán vào submission_xgb.csv thành công!
